# MMS adapter fine-tuning
based on https://huggingface.co/blog/mms_adapters

In [ ]:
%pip install -U pip
%pip install --no-cache-dir "transformers==4.57.1" accelerate "datasets[audio]" evaluate jiwer safetensors huggingface_hub tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 34.5 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 178.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 134.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 202.0 MB/s  0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [evaluate]


In [ ]:
from huggingface_hub import (login, notebook_login, create_repo, HfFolder, snapshot_download)

import torch
from datasets import load_dataset, Audio

import json
from pathlib import Path
import os
import gc
import shutil
import re
import time

from transformers import (
    Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor,
    Wav2Vec2ForCTC, TrainingArguments, Trainer, EarlyStoppingCallback
)

from dataclasses import dataclass
from typing import Dict, List, Union

import numpy as np
import jiwer
import pandas as pd

from safetensors.torch import save_file as safe_save_file
from transformers.models.wav2vec2.modeling_wav2vec2 import WAV2VEC2_ADAPTER_SAFE_FILE

from tqdm.auto import tqdm

In [ ]:
model_id = 'facebook/mms-1b-all'
target_lang = 'ckt'
dataset_repo_id = 'tadgeis/chukchi-asr-data-private'

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

True
Tesla T4


In [ ]:
notebook_login()

In [ ]:
hf_token = HfFolder.get_token()

if hf_token is None:
    raise ValueError('HF token was not found.')

In [ ]:
dataset_dict = load_dataset(dataset_repo_id, token=hf_token)
dataset_dict = dataset_dict.cast_column('audio', Audio(sampling_rate=16_000))

dataset_dict

README.md:   0%|          | 0.00/422 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/411M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/410M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/145M [00:00<?, ?B/s]

data/dev-00000-of-00001.parquet:   0%|          | 0.00/60.0M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating dev split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
        num_rows: 2714
    })
    test: Dataset({
        features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
        num_rows: 429
    })
    dev: Dataset({
        features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
        num_rows: 140
    })
})

In [ ]:
example = dataset_dict['train'][1000]

print(example.keys())
print(example['resource'])
print(example['path'])
print(example['sentence'])
print(example['audio']['sampling_rate'])
print(example['audio']['array'].shape)

dict_keys(['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'])
chuklang
I am from Chukotka_5.wav
мури вай амваанвыкэнайгым гым
16000
(42240,)


In [ ]:
vocab_source_dataset = dataset_dict['train']

def extract_all_chars(batch):
    all_text = ' '.join(batch['sentence'])
    vocab = sorted(list(set(all_text)))
    return {'vocab': [vocab], 'all_text': [all_text]}

vocab_result = vocab_source_dataset.map(extract_all_chars, batched=True,
    batch_size=-1, keep_in_memory=True,
    remove_columns=vocab_source_dataset.column_names)

vocab_list = vocab_result['vocab'][0]

print(vocab_list)
print('Number of raw characters:', len(vocab_list))

Map:   0%|          | 0/2714 [00:00<?, ? examples/s]

[' ', "'", 'а', 'б', 'в', 'г', 'д', 'е', 'ж', 'з', 'и', 'й', 'к', 'л', 'м', 'н', 'о', 'п', 'р', 'с', 'т', 'у', 'ф', 'х', 'ц', 'ч', 'ш', 'щ', 'ъ', 'ы', 'ь', 'э', 'ю', 'я', 'ё', 'ӄ', 'ӈ', 'ԓ']
Number of raw characters: 38


In [ ]:
vocab_dict = {char: idx for idx, char in enumerate(vocab_list)}

if ' ' not in vocab_dict:
    raise ValueError('Space character is not in vocabulary.')

vocab_dict['|'] = vocab_dict[' ']
del vocab_dict[' ']

vocab_dict['[UNK]'] = len(vocab_dict)
vocab_dict['[PAD]'] = len(vocab_dict)

In [ ]:
new_vocab_dict = {target_lang: vocab_dict}

with open('vocab.json', 'w', encoding='utf-8') as vocab_file:
    json.dump(new_vocab_dict, vocab_file, ensure_ascii=False, indent=2)

In [ ]:
tokenizer = Wav2Vec2CTCTokenizer.from_pretrained("./", unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|", target_lang=target_lang)

In [ ]:
feature_extractor = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16_000,
    padding_value=0.0, do_normalize=True, return_attention_mask=True)

processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

In [ ]:
def prepare_dataset(batch):

    audio = batch['audio']

    batch['input_values'] = processor(audio['array'], sampling_rate=audio['sampling_rate']).input_values[0]
    batch['input_length'] = len(batch['input_values'])

    batch['labels'] = processor(text=batch['sentence']).input_ids
    return batch

In [ ]:
@dataclass
class DataCollatorCTCWithPadding:
    """
    Data collator that will dynamically pad the inputs received.
    Args:
        processor (:class:`~transformers.Wav2Vec2Processor`)
            The processor used for proccessing the data.
        padding (:obj:`bool`, :obj:`str` or :class:`~transformers.tokenization_utils_base.PaddingStrategy`, `optional`, defaults to :obj:`True`):
            Select a strategy to pad the returned sequences (according to the model's padding side and padding index)
            among:
            * :obj:`True` or :obj:`'longest'`: Pad to the longest sequence in the batch (or no padding if only a single
              sequence if provided).
            * :obj:`'max_length'`: Pad to a maximum length specified with the argument :obj:`max_length` or to the
              maximum acceptable input length for the model if that argument is not provided.
            * :obj:`False` or :obj:`'do_not_pad'` (default): No padding (i.e., can output a batch with sequences of
              different lengths).
    """

    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need
        # different padding methods
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels_batch = self.processor.pad(
            labels=label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        batch["labels"] = labels

        return batch


def normalize_spaces(text):
    if pd.isna(text):
        return ''

    text = str(text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    # we do not want to group tokens when computing the metrics
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    pred_str = [normalize_spaces(text) for text in pred_str]
    label_str = [normalize_spaces(text) for text in label_str]

    wer = jiwer.wer(label_str, pred_str)
    cer = jiwer.cer(label_str, pred_str)

    return {'wer': wer, 'cer': cer}


def create_mms_adapter_model():
    model = Wav2Vec2ForCTC.from_pretrained(
        'facebook/mms-1b-all',
        attention_dropout=0.0,
        hidden_dropout=0.0,
        feat_proj_dropout=0.0,
        layerdrop=0.0,
        ctc_loss_reduction='mean',
        pad_token_id=processor.tokenizer.pad_token_id,
        vocab_size=len(processor.tokenizer),
        ignore_mismatched_sizes=True,
    )

    model.init_adapter_layers()
    model.freeze_base_model()

    adapter_weights = model._get_adapters()

    for param in adapter_weights.values():
        param.requires_grad = True

    return model


def prepare_resource_test_dataset(resource):
    test_dataset_raw_resource = dataset_dict['test'].filter(
        lambda example: example['resource'] == resource
    )

    if len(test_dataset_raw_resource) == 0:
        raise ValueError(f'No test examples found for resource: {resource}')

    test_dataset_resource = test_dataset_raw_resource.map(
        prepare_dataset,
        remove_columns=test_dataset_raw_resource.column_names,
        load_from_cache_file=False
    )

    decoded_labels = processor.batch_decode(test_dataset_resource['labels'], group_tokens=False)
    decoded_labels = [normalize_spaces(text) for text in decoded_labels]

    references = [normalize_spaces(text) for text in test_dataset_raw_resource['sentence']]
    mismatches = [
        (i, ref, label)
        for i, (ref, label) in enumerate(zip(references, decoded_labels))
        if ref != label
    ]

    print(f'{resource} mismatches before predict:', len(mismatches))

    if len(mismatches) > 0:
        print(mismatches[:5])
        raise ValueError(f'{resource}: raw references and prepared labels do not match')

    return test_dataset_raw_resource, test_dataset_resource



def predict_dataset_in_order(model, prepared_dataset, raw_dataset,
    data_collator, processor, batch_size=4):
    device = next(model.parameters()).device
    model.eval()

    rows = []

    for start in tqdm(range(0, len(prepared_dataset), batch_size)):
        end = min(start + batch_size, len(prepared_dataset))

        features = [prepared_dataset[i] for i in range(start, end)]

        batch = data_collator(features)

        input_batch = {key: value.to(device) for key, value in batch.items() if key != 'labels'}

        with torch.no_grad():
            logits = model(**input_batch).logits

        pred_ids = torch.argmax(logits, dim=-1)
        pred_str = processor.batch_decode(pred_ids)

        pred_str = [normalize_spaces(text) for text in pred_str]

        for local_i, prediction in enumerate(pred_str):
            raw_i = start + local_i
            raw_example = raw_dataset[raw_i]

            rows.append(
                {
                    'resource': raw_example['resource'],
                    'path': raw_example['path'],
                    'reference': normalize_spaces(raw_example['sentence']),
                    'prediction': prediction
                }
            )

    return pd.DataFrame(rows)


def evaluate_resource_test(resource, experiment_name, batch_size=4):
    test_dataset_raw_resource, test_dataset_resource = prepare_resource_test_dataset(
        resource
    )

    results_df = predict_dataset_in_order(
        model=trainer.model,
        prepared_dataset=test_dataset_resource,
        raw_dataset=test_dataset_raw_resource,
        data_collator=data_collator,
        processor=processor,
        batch_size=batch_size
    )

    wer = jiwer.wer(
        results_df['reference'].tolist(),
        results_df['prediction'].tolist()
    )

    cer = jiwer.cer(
        results_df['reference'].tolist(),
        results_df['prediction'].tolist()
    )

    predictions_path = f'mms_adapter_{experiment_name}_{resource}_test_predictions.csv'

    results_df.to_csv(
        predictions_path,
        index=False,
        encoding='utf-8-sig'
    )

    print(f'{resource}_test WER:', wer)
    print(f'{resource}_test CER:', cer)

    row = {
        'model': 'MMS-1b-all adapter fine-tuning',
        'training': experiment_name,
        'subset': f'{resource}_test',
        'WER': wer,
        'CER': cer,
        'n_files': len(results_df),
        'predictions_file': predictions_path
    }

    return row, results_df

In [ ]:
data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

# pooled
train (fine-tune) on bible_train + radio_train + chuklang_train

eval and test on chuklang_dev and chuklang_test respectively

In [ ]:
experiment_name = 'pooled'
model_repo_id = 'tadgeis/mms-1b-ckt-pooled'
resources_to_evaluate = ['chuklang', 'radio', 'bible']

create_repo(repo_id=model_repo_id, repo_type='model', private=False, exist_ok=True, token=hf_token)

processor.push_to_hub(model_repo_id, private=False, token=hf_token)

SEED = 42

train_dataset_raw = dataset_dict['train'].filter(lambda example: example['resource'] in ['bible', 'radio', 'chuklang'])
eval_dataset_raw = dataset_dict['dev'].filter(lambda example: example['resource'] == 'chuklang')

train_dataset_raw = train_dataset_raw.shuffle(seed=SEED)

print(train_dataset_raw)
print(eval_dataset_raw)

train_dataset = train_dataset_raw.map(prepare_dataset, remove_columns=train_dataset_raw.column_names, load_from_cache_file=False)
eval_dataset = eval_dataset_raw.map(prepare_dataset, remove_columns=eval_dataset_raw.column_names, load_from_cache_file=False)

print(train_dataset)
print(eval_dataset)

README.md: 0.00B [00:00, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


Filter:   0%|          | 0/2714 [00:00<?, ? examples/s]

Filter:   0%|          | 0/140 [00:00<?, ? examples/s]

Dataset({
    features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
    num_rows: 2714
})
Dataset({
    features: ['resource', 'path', 'sentence', 'duration', 'source_id', 'duration_bin', 'split', 'audio'],
    num_rows: 140
})


Map:   0%|          | 0/2714 [00:00<?, ? examples/s]

Map:   0%|          | 0/140 [00:00<?, ? examples/s]

Dataset({
    features: ['input_values', 'input_length', 'labels'],
    num_rows: 2714
})
Dataset({
    features: ['input_values', 'input_length', 'labels'],
    num_rows: 140
})


In [ ]:
model = create_mms_adapter_model()
output_dir = model_repo_id.split('/')[-1]

trainable_params = sum(param.numel() for param in model.parameters() if param.requires_grad)
all_params = sum(param.numel() for param in model.parameters())

print(f'Trainable params: {trainable_params:,}')
print(f'All params: {all_params:,}')
print(f'Trainable share: {100 * trainable_params / all_params:.4f}%')

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/mms-1b-all and are newly initialized because the shapes did not match:
- lm_head.bias: found shape torch.Size([154]) in the checkpoint and torch.Size([42]) in the model instantiated
- lm_head.weight: found shape torch.Size([154, 1280]) in the checkpoint and torch.Size([42, 1280]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable params: 2,204,970
All params: 964,702,378
Trainable share: 0.2286%


In [ ]:
# КОД ДЛЯ ПРОДОЛЖЕНИЯ С ТЕКУЩЕГО ПОСЛЕДНЕГО ЧЕКПОИНТА
# snapshot_download(repo_id=model_repo_id, repo_type='model', token=hf_token, local_dir=output_dir)

# resume_checkpoint = str(Path(output_dir) / 'last-checkpoint')

# print(resume_checkpoint)
# print(Path(resume_checkpoint).exists())

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

last-checkpoint/optimizer.pt:   0%|          | 0.00/17.9M [00:00<?, ?B/s]

last-checkpoint/model.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/256 [00:00<?, ?B/s]

...

training_args.bin:   0%|          | 0.00/5.91k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json:   0%|          | 0.00/572 [00:00<?, ?B/s]

mms-1b-ckt-pooled/last-checkpoint
True


In [ ]:
training_args = TrainingArguments(
    output_dir=output_dir,

    group_by_length=True,

    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,

    eval_strategy='steps',
    save_strategy='steps',

    num_train_epochs=12,

    gradient_checkpointing=True,
    fp16=torch.cuda.is_available(),

    save_steps=50,
    eval_steps=50,
    logging_steps=50,

    learning_rate=1e-3,
    warmup_steps=25,

    save_total_limit=4,

    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,

    push_to_hub=True,
    hub_model_id=model_repo_id,
    hub_private_repo=False,
    hub_token=hf_token,
    hub_strategy='checkpoint',
    hub_always_push=True,

    report_to='none',
    disable_tqdm=False
)


trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=processor.feature_extractor,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=3,
            early_stopping_threshold=0.001
        )
    ]
)

/tmp/ipykernel_1359/1160616290.py:42: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

# trainer.train(resume_from_checkpoint=resume_checkpoint)  # ВАРИАНТ ДЛЯ ПРОДОЛЖЕНИЯ С СУЩЕСТВУЮЩЕГО ПОСЛЕДНЕГО ЧЕКПОИНТА

	per_device_train_batch_size: 16 (from args) != 4 (from trainer_state.json)


Step,Training Loss,Validation Loss,Wer,Cer
450,0.553300,0.874180,0.882540,0.221511
500,0.820600,0.841097,0.855556,0.215840
550,0.540300,0.917480,0.873016,0.235595


Could not locate the best model at mms-1b-ckt-pooled/checkpoint-400/pytorch_model.bin, if you are running a distributed training on multiple nodes, you should activate `--save_on_each_node`.


TrainOutput(global_step=550, training_loss=0.17401889107444068, metrics={'train_runtime': 1585.3924, 'train_samples_per_second': 20.543, 'train_steps_per_second': 5.139, 'total_flos': 5.245803603052858e+18, 'train_loss': 0.17401889107444068, 'epoch': 0.8100147275405007})

In [ ]:
log_history_df = pd.DataFrame(trainer.state.log_history)

log_history_df.to_csv(f'mms_adapter_{experiment_name}_log_history.csv', index=False, encoding='utf-8-sig')

print('Best checkpoint:', trainer.state.best_model_checkpoint)
print('Best metric:', trainer.state.best_metric)

Best checkpoint: mms-1b-ckt-pooled/checkpoint-400
Best metric: 0.2134625937442839


In [ ]:
best_checkpoint = trainer.state.best_model_checkpoint
best_dev_cer = trainer.state.best_metric

best_step = None
best_dev_loss = None
best_dev_wer = None

if best_checkpoint is not None:
    match = re.search(r'checkpoint-(\d+)', best_checkpoint)

    if match is not None:
        best_step = int(match.group(1))

        best_eval_rows = log_history_df[
            (log_history_df['step'] == best_step) &
            (log_history_df['eval_cer'].notna())
        ]

        if len(best_eval_rows) > 0:
            best_eval_row = best_eval_rows.iloc[0]
            best_dev_loss = best_eval_row['eval_loss']
            best_dev_wer = best_eval_row['eval_wer']
            best_dev_cer = best_eval_row['eval_cer']

In [ ]:
summary_rows = []
prediction_dfs = {}

for resource in resources_to_evaluate:
    row, results_df = evaluate_resource_test(resource=resource, experiment_name=experiment_name, batch_size=16)

    row['best_dev_checkpoint'] = best_checkpoint
    row['best_dev_step'] = best_step
    row['best_dev_loss'] = best_dev_loss
    row['best_dev_WER'] = best_dev_wer
    row['best_dev_CER'] = best_dev_cer

    summary_rows.append(row)
    prediction_dfs[resource] = results_df

summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(f'mms_adapter_{experiment_name}_results_summary.csv', index=False, encoding='utf-8-sig')

summary_df

Filter:   0%|          | 0/429 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

chuklang mismatches before predict: 0


  0%|          | 0/50 [00:00<?, ?it/s]

chuklang_test WER: 0.8942420681551116
chuklang_test CER: 0.2521054811542179


...

Map:   0%|          | 0/146 [00:00<?, ? examples/s]

bible mismatches before predict: 0


  0%|          | 0/37 [00:00<?, ?it/s]

bible_test WER: 0.3217031342400946
bible_test CER: 0.04448071822077127


,model,training,subset,WER,CER,n_files,predictions_file,best_dev_checkpoint,best_dev_step,best_dev_loss,best_dev_WER,best_dev_CER
0,MMS-1b-all adapter fine-tuning,pooled,chuklang_test,0.894242,0.252105,200,mms_adapter_pooled_chuklang_test_predictions.csv,mms-1b-ckt-pooled/checkpoint-400,400,0.812787,0.833333,0.213463
1,MMS-1b-all adapter fine-tuning,pooled,radio_test,0.736983,0.166379,83,mms_adapter_pooled_radio_test_predictions.csv,mms-1b-ckt-pooled/checkpoint-400,400,0.812787,0.833333,0.213463
2,MMS-1b-all adapter fine-tuning,pooled,bible_test,0.321703,0.044481,146,mms_adapter_pooled_bible_test_predictions.csv,mms-1b-ckt-pooled/checkpoint-400,400,0.812787,0.833333,0.213463


In [ ]:
prediction_dfs['chuklang'].head()

,resource,path,reference,prediction
0,chuklang,A chatterbox and a wanton girl_2.wav,ӄоле итгъэт ӄынвэтэ ӈиръэ ӈэвысӄэтти элерэты н...,ӄоԓэ итгъатӄынгыт ниръэ ӈэвычӄатэ элерэтынатанат
1,chuklang,A chatterbox and a wanton girl_3.wav,ӄол вэтгавӈавъым ӄол камэлгыӈав,ӄоԓ вэтгёгыӈаым ӄоԓкамэлгыӈа
2,chuklang,A chatterbox and a wanton girl_4.wav,ынкы илирыкы нантыӈӈонатъым,ынкы илирык нантыӈӈонатъым
3,chuklang,Abramovich_4.wav,гэчевкы нынтыӄин таӈколё ынкы ныгынритӄин,эчевкынынтыӄэтаӈколёмкыныгынӈитки
4,chuklang,An evil spirit and a dicky bird_1.wav,энмэн гатвален каԓьайӈын ынкъам пчеӄалгын,энмэ гэтвалин кальан ынкъам пчекаԓг


In [ ]:
adapter_file = WAV2VEC2_ADAPTER_SAFE_FILE.format(target_lang)
adapter_file = os.path.join(training_args.output_dir, adapter_file)

adapter_weights = {
    name: tensor.detach().cpu()
    for name, tensor in trainer.model._get_adapters().items()
}

safe_save_file(adapter_weights, adapter_file, metadata={'format': 'pt'})

print(adapter_file)

processor.save_pretrained(training_args.output_dir)
trainer.save_model(training_args.output_dir)

trainer.push_to_hub()
processor.push_to_hub(model_repo_id, private=False, token=hf_token)

mms-1b-ckt-pooled/adapter.ckt.safetensors


In [ ]:
files_to_download = [
    Path(f'mms_adapter_{experiment_name}_log_history.csv'),
    Path(f'mms_adapter_{experiment_name}_results_summary.csv'),
]

for resource in resources_to_evaluate:
    files_to_download.append(
        Path(f'mms_adapter_{experiment_name}_{resource}_test_predictions.csv')
    )

for path in files_to_download:
    if path.exists():
        print(f'Ready to download: {path}')
    else:
        print(f'File not found: {path}')